<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Vector_databases_and_RAG_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#some packagesd are missing. We need to install those
%pip install -q sentence-transformers faiss-cpu chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [ ]:
import numpy as np
import faiss
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
#First create the vector databases
faq_documents = (
    "Refunds are only available if you request cancellation within the first 7 days of the course start date. After that, no reimbursement is provided."
    "All live sessions are recorded and uploaded to the learning portal within 24 hours, so you can watch them later if you miss a class."
    "The course covers Python programming, SQL for data analysis, statistics fundamentals, and an introduction to machine learning."
    "You need a laptop with at least 8GB RAM. No prior coding experience is required to join this program."
    "Certificates are issued only after you complete at least 80% of the live sessions and submit the final capstone project."
    "You can reach the support team via email at support@example.com or through the live chat on the learning portal, available 9 AM to 7 PM on weekdays."
    "The capstone project involves analyzing a real-world dataset and presenting your findings in a short report and a 10-minute video presentation."
    "Doubt-clearing sessions are held every Saturday at 5 PM, and attendance is optional but recommended before exams."
)

print(f"We have {len(faq_documents)} FAQ documents in our knowledge base.")


We have 1029 FAQ documents in our knowledge base.


In [ ]:
#convert that into chunks. Here, you may use the sentence level chunks

import re

total_sen=len(faq_documents)

def sentence_level(text):
  '''Split sentences by punctuation rules'''
  chunk=re.split(r'(?<=[.!?])\s+', text.strip())
  return [s for s in chunk if s]

chunks_1=[]
sen_chunk= sentence_level(faq_documents)
for i,c in enumerate(sen_chunk):
  print(f"[Chunk {i}] : {c}")
  chunks_1.append(c)

[Chunk 0] : Refunds are only available if you request cancellation within the first 7 days of the course start date.
[Chunk 1] : After that, no reimbursement is provided.All live sessions are recorded and uploaded to the learning portal within 24 hours, so you can watch them later if you miss a class.The course covers Python programming, SQL for data analysis, statistics fundamentals, and an introduction to machine learning.You need a laptop with at least 8GB RAM.
[Chunk 2] : No prior coding experience is required to join this program.Certificates are issued only after you complete at least 80% of the live sessions and submit the final capstone project.You can reach the support team via email at support@example.com or through the live chat on the learning portal, available 9 AM to 7 PM on weekdays.The capstone project involves analyzing a real-world dataset and presenting your findings in a short report and a 10-minute video presentation.Doubt-clearing sessions are held every Saturday 

In [ ]:
chunks_1

['Refunds are only available if you request cancellation within the first 7 days of the course start date.',
 'After that, no reimbursement is provided.All live sessions are recorded and uploaded to the learning portal within 24 hours, so you can watch them later if you miss a class.The course covers Python programming, SQL for data analysis, statistics fundamentals, and an introduction to machine learning.You need a laptop with at least 8GB RAM.',
 'No prior coding experience is required to join this program.Certificates are issued only after you complete at least 80% of the live sessions and submit the final capstone project.You can reach the support team via email at support@example.com or through the live chat on the learning portal, available 9 AM to 7 PM on weekdays.The capstone project involves analyzing a real-world dataset and presenting your findings in a short report and a 10-minute video presentation.Doubt-clearing sessions are held every Saturday at 5 PM, and attendance is

In [ ]:
#embedding the above list of chunks into vectors
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(chunks_1)
print("Shape of embeddings: ",embeddings.shape)
print("Each sentence became the vector of",embeddings.shape[1],"numbers")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape of embeddings:  (3, 384)
Each sentence became the vector of 384 numbers


In [ ]:
from numpy.linalg import norm

def cosine_similarity(a, b):
  return np.dot(a, b)/(norm(a)*norm(b))

print("The cosine similiarity between two consecutive vectors:",cosine_similarity(embeddings[0],embeddings[1]))
print("The cosine similiarity between two consecutive vectors:",cosine_similarity(embeddings[0],embeddings[2]))

The cosine similiarity between two consecutive vectors: 0.40142852
The cosine similiarity between two consecutive vectors: 0.25159475


In [ ]:
emb_dim=embeddings.shape[1]
faiss.normalize_L2(embeddings)
faiss_index = faiss.IndexFlatIP(emb_dim)
faiss_index.add(embeddings)

In [ ]:
def search_faiss(query, top_k=2):
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)
    scores, indices = faiss_index.search(query_vector, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        # Use chunks_1 list to map the index back to the full sentence
        results.append((chunks_1[idx], float(score)))
    return results

query = "Can I get my money back if I change my mind?"
for doc, score in search_faiss(query):
    print(f"(score={score:.3f}) {doc}\n")

(score=0.352) Refunds are only available if you request cancellation within the first 7 days of the course start date.

(score=0.087) After that, no reimbursement is provided.All live sessions are recorded and uploaded to the learning portal within 24 hours, so you can watch them later if you miss a class.The course covers Python programming, SQL for data analysis, statistics fundamentals, and an introduction to machine learning.You need a laptop with at least 8GB RAM.



In [ ]:
CONFIDENCE_THRESHOLD = 0.35  # below this similarity score, we don't trust the match

def faq_bot(question, top_k=1):
    results = search_faiss(question, top_k=top_k)
    best_doc, best_score = results[0]

    print(f"Question: {question}")
    print(f"Top match (score={best_score:.3f}): {best_doc}")

    if best_score >= CONFIDENCE_THRESHOLD:
        print(f"\n🤖 Bot answer: {best_doc}")
    else:
        print("\n🤖 Bot answer: I'm not confident I have information about that. "
              "Please contact support@example.com.")
    print("-" * 80)

faq_bot("How do I get a certificate?")
faq_bot("What laptop do I need?")

Question: How do I get a certificate?
Top match (score=0.387): f

🤖 Bot answer: f
--------------------------------------------------------------------------------
Question: What laptop do I need?
Top match (score=0.238): e

🤖 Bot answer: I'm not confident I have information about that. Please contact support@example.com.
--------------------------------------------------------------------------------


In [ ]:
import numpy as np
from numpy.linalg import norm

documents = [
    {"id": "FAQ1", "topic": "refund", "text": "Refunds are available within 7 days of purchase.", "embedding": [1.0, 0.0, 0.0]},
    {"id": "FAQ2", "topic": "schedule", "text": "Live classes run every Tuesday and Thursday.", "embedding": [0.0, 1.0, 0.0]},
    {"id": "FAQ3", "topic": "refund", "text": "Refund requests are processed within 21 working days.", "embedding": [0.8, 0.6, 0.0]},
    {"id": "FAQ4", "topic": "curriculum", "text": "Module 2 covers SQL fundamentals.", "embedding": [0.0, 0.0, 1.0]},
    {"id": "FAQ5", "topic": "payment", "text": "Tuition can be paid in three monthly installments.", "embedding": [0.6, 0.0, 0.8]},
]

collection = []

def cosine_similarity(a, b):
  return np.dot(a, b)/(norm(a)*norm(b))

def add_documents(docs):
    # TODO: append each document dict from `docs` into the `collection` list.
    collection.extend(docs)

def query(query_embedding, n_results=2):
    scores = []
    for doc in collection:
        # TODO: compute the cosine similarity between query_embedding and
        # doc["embedding"] using np.dot.
        similarity = cosine_similarity(query_embedding, doc["embedding"])
        scores.append({"id": doc["id"], "text": doc["text"], "score": similarity})
    # TODO: sort `scores` by "score" descending and return only the top n_results.
    scores.sort(key=lambda x: x["score"], reverse=True)
    return scores[:n_results]

if __name__ == "__main__":
    # Reset collection first to avoid duplicates from previous run
    collection = []
    add_documents(documents)
    print(query([0.8, 0.6, 0.0], n_results=2))

[{'id': 'FAQ3', 'text': 'Refund requests are processed within 21 working days.', 'score': np.float64(1.0)}, {'id': 'FAQ1', 'text': 'Refunds are available within 7 days of purchase.', 'score': np.float64(0.8)}]


In [ ]:
import re

text = "Refunds are only available within the first seven days of enrollment. Live classes run every Tuesday and Thursday at 8 pm. The curriculum covers Python, SQL, and statistics."

def sentence_chunk(document):
  '''Split sentences by punctuation rules'''
  chunk=re.split(r'(?<=[.!?])\s+', text.strip())
  return [s for s in chunk if s]


if __name__ == "__main__":
    for chunk in sentence_chunk(text):
        print(chunk)


Refunds are only available within the first seven days of enrollment.
Live classes run every Tuesday and Thursday at 8 pm.
The curriculum covers Python, SQL, and statistics.


In [ ]:
import numpy as np

documents = [
    {"id": "E1", "text": "Refunds are only available within the first seven days of enrollment.", "embedding": [1.0, 0.0, 0.0]},
    {"id": "E2", "text": "Doubt-clearing sessions run every Saturday at 5 pm.", "embedding": [0.0, 1.0, 0.0]},
    {"id": "E3", "text": "Module 3 covers statistics and hypothesis testing.", "embedding": [0.0, 0.0, 1.0]},
    {"id": "E4", "text": "Refund requests are processed within 21 working days.", "embedding": [0.8, 0.6, 0.0]},
]
THRESHOLD = 0.30
REFUSAL = {"refused": True, "message": "I don't have enough information about that; please contact the support team."}

def search(query_embedding):
    best_doc = None
    best_score = None
    # TODO: loop over `documents`, compute the dot product between
    # query_embedding and each doc["embedding"], and track the doc with the
    # highest score in best_doc / best_score.
    for doc in documents:
        similarity = np.dot(query_embedding, doc["embedding"])
        if best_score is None or similarity > best_score:
            best_doc = doc
            best_score = similarity

    # TODO: if best_score is below THRESHOLD, return REFUSAL.
    # TODO: otherwise, return {"id": best_doc["id"], "text": best_doc["text"], "score": best_score}.
    if best_score < THRESHOLD:
        return REFUSAL
    else:
        return {"id": best_doc["id"], "text": best_doc["text"], "score": best_score}

if __name__ == "__main__":
    print(search([0.8, 0.6, 0.0]))
    print(search([-0.2, 0.1, -0.1]))

{'id': 'E4', 'text': 'Refund requests are processed within 21 working days.', 'score': np.float64(1.0)}
{'refused': True, 'message': "I don't have enough information about that; please contact the support team."}
